# 2. Data Engineering

ตรวจสอบคุณภาพข้อมูลและตัดคอลัมน์ที่ไม่ควรใช้เป็น input feature ก่อนแบ่งข้อมูล

In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("../data/raw/dataset.csv")
df.shape

(800, 25)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Student_ID              800 non-null    object 
 1   Age                     800 non-null    int64  
 2   Gender                  770 non-null    object 
 3   Year_of_Study           800 non-null    int64  
 4   Department              777 non-null    object 
 5   Residence_Type          783 non-null    object 
 6   Attendance_Percent      749 non-null    float64
 7   Study_Hours_Per_Day     765 non-null    float64
 8   Previous_GPA            765 non-null    float64
 9   Backlogs                785 non-null    float64
 10  Sleep_Hours             765 non-null    float64
 11  Screen_Time_Hours       764 non-null    float64
 12  Exercise_Freq_Per_Week  800 non-null    int64  
 13  Social_Activity_Score   760 non-null    float64
 14  Part_Time_Job           775 non-null    ob

In [13]:
df.isnull().sum()

Student_ID                 0
Age                        0
Gender                    30
Year_of_Study              0
Department                23
Residence_Type            17
Attendance_Percent        51
Study_Hours_Per_Day       35
Previous_GPA              35
Backlogs                  15
Sleep_Hours               35
Screen_Time_Hours         36
Exercise_Freq_Per_Week     0
Social_Activity_Score     40
Part_Time_Job             25
Family_Income_Bracket     13
Financial_Stress_Score    43
Family_Support_Score      44
Stress_Level              49
Anxiety_Score             39
Motivation_Score          41
Peer_Pressure_Score       41
Counseling_Access          0
Burnout_Level              0
Dropout_Risk               0
dtype: int64

In [14]:
categorical_cols = [
    "Gender", "Department", "Residence_Type", "Part_Time_Job",
    "Family_Income_Bracket", "Counseling_Access", "Burnout_Level", "Dropout_Risk"
]

for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


Gender
Gender
Female    392
Male      346
Other      32
NaN        30
Name: count, dtype: int64

Department
Department
Engineering    209
Business       151
Science        143
Arts           136
Medicine        77
Law             61
NaN             23
Name: count, dtype: int64

Residence_Type
Residence_Type
Hostel         290
Day Scholar    286
PG/Rented      207
NaN             17
Name: count, dtype: int64

Part_Time_Job
Part_Time_Job
No     547
Yes    228
NaN     25
Name: count, dtype: int64

Family_Income_Bracket
Family_Income_Bracket
Middle          242
Lower-Middle    227
Upper-Middle    131
Low              95
High             92
NaN              13
Name: count, dtype: int64

Counseling_Access
Counseling_Access
No     535
Yes    265
Name: count, dtype: int64

Burnout_Level
Burnout_Level
Low       320
Medium    280
High      200
Name: count, dtype: int64

Dropout_Risk
Dropout_Risk
Yes    437
No     363
Name: count, dtype: int64


In [16]:
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated Student_ID:", df["Student_ID"].duplicated().sum())

Duplicated rows: 0
Duplicated Student_ID: 0


## Feature exclusion decisions

- **Student_ID** เป็น unique identifier จึงไม่ควรให้โมเดลเรียนรู้ และอาจทำให้โมเดลจำตัวอย่างแทนที่จะเรียนรู้รูปแบบทั่วไป
- **Gender** ถูกตัดออกเพราะ bias
- **Burnout_Level** ถูกตัดออกเพราะใน dataset นี้เป็นอีก target outcome
- เพื่อให้ช้งานง่ายขึ้น ตัด **Sleep_Hours, Exercise_Freq_Per_Week, Social_Activity_Score และ Peer_Pressure_Score** ออก โดยยังคง feature กลุ่ม Academic ไว้ทั้งหมด และลดจำนวน input ที่ผู้ใช้ต้องกรอก

หลังการตัด จะเหลือ **17 input features** สำหรับโมเดล

In [17]:
drop_cols = [
    "Student_ID",
    "Gender",
    "Burnout_Level",
    "Sleep_Hours",
    "Exercise_Freq_Per_Week",
    "Social_Activity_Score",
    "Peer_Pressure_Score",
]

df = df.drop(columns=drop_cols)
print("Remaining input features:", df.shape[1] - 1)
df.columns.tolist()

Remaining input features: 17


['Age',
 'Year_of_Study',
 'Department',
 'Residence_Type',
 'Attendance_Percent',
 'Study_Hours_Per_Day',
 'Previous_GPA',
 'Backlogs',
 'Screen_Time_Hours',
 'Part_Time_Job',
 'Family_Income_Bracket',
 'Financial_Stress_Score',
 'Family_Support_Score',
 'Stress_Level',
 'Anxiety_Score',
 'Motivation_Score',
 'Counseling_Access',
 'Dropout_Risk']

In [18]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
df.to_parquet("../data/processed/student_data_pre_impute.parquet", index=False)
print(df.shape)

(800, 18)
